<a href="https://colab.research.google.com/github/gbcancian/data_analysis_treinos/blob/main/Academia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Objetivos do Projeto

1. Quantidade de treinos por semana
2. Exercícios mais utilizados
3. Quantidade média de repetições por exercício
4. Evolução de carga (Progressive overload)
5. Volume total de treino
6. Volume por treino
7. Tempo médio de treino

#Tratamento dos Dados


In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

treinos = pd.read_csv('workouts.csv')

In [3]:
#Renomeando as colunas do DataFrame
treinos.columns=['titulo','data_inicio','data_final','descricao','exercicio','superset_id','notas_exercicio','indice_serie','tipo_serie','peso','repeticoes','distancia','duracao','rpe']
treinos.head()

,titulo,data_inicio,data_final,descricao,exercicio,superset_id,notas_exercicio,indice_serie,tipo_serie,peso,repeticoes,distancia,duracao,rpe
0,Terc - Pull,"24 Mar 2026, 16:37","24 Mar 2026, 18:19",NaN,Bent Over Row (Barbell),NaN,NaN,0,normal,40.0,12.0,NaN,NaN,NaN
1,Terc - Pull,"24 Mar 2026, 16:37","24 Mar 2026, 18:19",NaN,Bent Over Row (Barbell),NaN,NaN,1,normal,60.0,10.0,NaN,NaN,NaN
2,Terc - Pull,"24 Mar 2026, 16:37","24 Mar 2026, 18:19",NaN,Bent Over Row (Barbell),NaN,NaN,2,normal,60.0,11.0,NaN,NaN,NaN
3,Terc - Pull,"24 Mar 2026, 16:37","24 Mar 2026, 18:19",NaN,Bent Over Row (Barbell),NaN,NaN,3,normal,70.0,8.0,NaN,NaN,NaN
4,Terc - Pull,"24 Mar 2026, 16:37","24 Mar 2026, 18:19",NaN,Rope Straight Arm Pulldown,NaN,NaN,0,normal,10.0,12.0,NaN,NaN,NaN


In [4]:
#Verificando se possui pesos nulos
treinos.query('peso == 0 | peso < 0 | peso.isnull()')

,titulo,data_inicio,data_final,descricao,exercicio,superset_id,notas_exercicio,indice_serie,tipo_serie,peso,repeticoes,distancia,duracao,rpe
45,Seg - Push,"23 Mar 2026, 16:19","23 Mar 2026, 17:52",NaN,Chest Dip,NaN,NaN,0,normal,NaN,20.0,NaN,0.0,NaN
46,Seg - Push,"23 Mar 2026, 16:19","23 Mar 2026, 17:52",NaN,Chest Dip,NaN,NaN,1,normal,NaN,15.0,NaN,0.0,NaN
47,Seg - Push,"23 Mar 2026, 16:19","23 Mar 2026, 17:52",NaN,Chest Dip,NaN,NaN,2,normal,NaN,12.0,NaN,0.0,NaN
48,Seg - Push,"23 Mar 2026, 16:19","23 Mar 2026, 17:52",NaN,Leg Press (Machine),NaN,NaN,0,normal,NaN,0.0,NaN,0.0,NaN
49,Seg - Push,"23 Mar 2026, 16:19","23 Mar 2026, 17:52",NaN,Leg Press (Machine),NaN,NaN,1,normal,NaN,0.0,NaN,0.0,NaN
50,Seg - Push,"23 Mar 2026, 16:19","23 Mar 2026, 17:52",NaN,Leg Press (Machine),NaN,NaN,2,normal,NaN,0.0,NaN,0.0,NaN
74,Terc - Pull,"18 Mar 2026, 18:36","18 Mar 2026, 20:32",NaN,Face Pull,NaN,NaN,2,normal,NaN,12.0,NaN,0.0,NaN
96,Seg - Push,"17 Mar 2026, 17:04","17 Mar 2026, 18:34",NaN,Chest Dip,NaN,NaN,0,normal,NaN,17.0,NaN,0.0,NaN
97,Seg - Push,"17 Mar 2026, 17:04","17 Mar 2026, 18:34",NaN,Chest Dip,NaN,NaN,0,normal,NaN,13.0,NaN,0.0,NaN
113,Seg - Push,"9 Mar 2026, 17:05","9 Mar 2026, 18:31",NaN,Butterfly (Pec Deck),NaN,NaN,0,normal,NaN,0.0,NaN,0.0,NaN


In [5]:
#Visualizando quais colunas não tem dados
treinos.query('descricao.notnull()')
treinos.query('superset_id.notnull()')
treinos.query('notas_exercicio.notnull()')
treinos.query('distancia.notnull()')
treinos.query('duracao != 0 & duracao.notnull()') #manter duração
treinos.query('rpe != 0 & rpe.notnull()')

,titulo,data_inicio,data_final,descricao,exercicio,superset_id,notas_exercicio,indice_serie,tipo_serie,peso,repeticoes,distancia,duracao,rpe


In [6]:
#Criando uma variavel com as colunas utilizaveis
treinos = treinos[['titulo','data_inicio','data_final','exercicio','indice_serie','tipo_serie','peso','repeticoes','duracao']]

In [7]:
#DataFrame tratado
treinos_validos = treinos.query('peso > 0')
treinos_validos.head()
treinos_validos.info()

<class 'pandas.core.frame.DataFrame'>
Index: 379 entries, 0 to 401
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   titulo        379 non-null    object 
 1   data_inicio   379 non-null    object 
 2   data_final    379 non-null    object 
 3   exercicio     379 non-null    object 
 4   indice_serie  379 non-null    int64  
 5   tipo_serie    379 non-null    object 
 6   peso          379 non-null    float64
 7   repeticoes    379 non-null    float64
 8   duracao       216 non-null    float64
dtypes: float64(3), int64(1), object(5)
memory usage: 29.6+ KB


In [8]:
#Formantando datas para o formato padrão do pandas
treinos_validos['data_inicio'] = pd.to_datetime(treinos_validos['data_inicio'], format='%d %b %Y, %H:%M', dayfirst=True, errors='coerce')
treinos_validos['data_final'] = pd.to_datetime(treinos_validos['data_final'], format='%d %b %Y, %H:%M', dayfirst=True, errors='coerce')
treinos_por_dia = treinos_validos.groupby('data_inicio')
treinos_por_dia.head()

/tmp/ipykernel_9834/1337660185.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  treinos_validos['data_inicio'] = pd.to_datetime(treinos_validos['data_inicio'], format='%d %b %Y, %H:%M', dayfirst=True, errors='coerce')
/tmp/ipykernel_9834/1337660185.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  treinos_validos['data_final'] = pd.to_datetime(treinos_validos['data_final'], format='%d %b %Y, %H:%M', dayfirst=True, errors='coerce')


,titulo,data_inicio,data_final,exercicio,indice_serie,tipo_serie,peso,repeticoes,duracao
0,Terc - Pull,2026-03-24 16:37:00,2026-03-24 18:19:00,Bent Over Row (Barbell),0,normal,40.0,12.0,NaN
1,Terc - Pull,2026-03-24 16:37:00,2026-03-24 18:19:00,Bent Over Row (Barbell),1,normal,60.0,10.0,NaN
2,Terc - Pull,2026-03-24 16:37:00,2026-03-24 18:19:00,Bent Over Row (Barbell),2,normal,60.0,11.0,NaN
3,Terc - Pull,2026-03-24 16:37:00,2026-03-24 18:19:00,Bent Over Row (Barbell),3,normal,70.0,8.0,NaN
4,Terc - Pull,2026-03-24 16:37:00,2026-03-24 18:19:00,Rope Straight Arm Pulldown,0,normal,10.0,12.0,NaN
...,...,...,...,...,...,...,...,...,...
383,Costas + Triceps,2024-08-05 16:18:00,2024-08-05 18:09:00,Triceps Rope Pushdown,3,normal,21.0,9.0,0.0
384,Costas + Triceps,2024-08-05 16:18:00,2024-08-05 18:09:00,Triceps Extension (Cable),3,normal,15.0,10.0,0.0
385,Costas + Triceps,2024-08-05 16:18:00,2024-08-05 18:09:00,Skullcrusher (Barbell),3,normal,10.0,10.0,0.0
386,Costas + Triceps,2024-08-05 16:18:00,2024-08-05 18:09:00,Chin Up (Assisted),3,normal,12.0,9.0,0.0


#Criação das Visualizações

## 4. Evolução de carga (Overload)



In [56]:
overload = (
treinos_validos.groupby(['exercicio', 'data_inicio'])
['peso'].max()
.reset_index()
)

overload = overload.sort_values(by=['data_inicio'])
overload.head()

,exercicio,data_inicio,peso
24,Chin Up (Assisted),2024-08-05 16:18:00,12.0
50,Landmine Row,2024-08-05 16:18:00,25.0
81,Seated Cable Row - Bar Grip,2024-08-05 16:18:00,27.5
4,Back Extension (Weighted Hyperextension),2024-08-05 16:18:00,57.5
102,Triceps Extension (Cable),2024-08-05 16:18:00,15.0


## 2. Exercicios mais utilizados (Série)

In [57]:
exercicios_mais_utilizados_serie = (
    treinos_validos
    .groupby(['exercicio'])
    .size()
    .reset_index(name='Quantidade Total Series')
)

exercicios_mais_utilizados_serie.columns = ['Exercicio', 'Quantidade Total Series']
exercicios_mais_utilizados_serie.sort_values(by='Quantidade Total Series', ascending=False)


,Exercicio,Quantidade Total Series
24,Lateral Raise (Dumbbell),25
14,Face Pull,22
4,Bent Over Row (Barbell),20
21,Lat Pulldown (Cable),17
26,Leg Press (Machine),16
0,Arnold Press (Dumbbell),16
46,Triceps Rope Pushdown,14
11,Crunch (Machine),12
12,Dumbbell Row,12
3,Bench Press (Dumbbell),12
